# 38 - Seasonal Precipitation Percentile Heatmap Analysis

## Input Data
- **Zarr stores** with 15-minute precipitation statistics by percentile band (seasonal subsets; we chose one)
  - Distribution data (PRECIP_PERIOD): Count of 15-min periods in each percentile band
  - Accumulation data (PRECIP_SUM): Total precipitation accumulation per band
- **Epochs**: HIST (1990-2005), MID (2050-2065), EOC (2085-2100)
- **Scenario**: RCP4.5 and RCP8.5 (we chose one)
- **Seasons**: DJF, MAM, JJA, SON
- **Spatial**: 7 NCA regions with regional masks
- **Format**: Compressed Zarr datasets

## Processing Steps
1. **Load masked regional data**: Apply NCA region masks to seasonal precipitation datasets
2. **Calculate percentile band statistics**:
   - Temporal sum for each grid cell (across seasonal time periods)
   - Spatial average across region (equal weight per grid cell)
3. **Compute percent changes**: Future epochs vs. historical baseline
4. **Generate visualizations**:
   - 2×2 heatmap grid 

## Output
- **Bar chart figures**: Individual region files + combined master figure (seasonal suffix)
- **Heatmap figure**: 2×2 grid (All 4 seasons within each MID and EOC)

In [2]:
# Imports and Configuration
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as path_effects
import pandas as pd
import xarray as xr
from scipy.stats import mannwhitneyu

matplotlib.use('Agg')
matplotlib.rcdefaults()
plt.close('all')

regions = ["Midwest", "Northeast", "NorthernGreatPlains", "Northwest",
           "Southeast", "SouthernGreatPlains", "Southwest"]

scenarios = [
    {"name": "4.5", "mid_suffix": "mid4p5", "eoc_suffix": "eoc4p5"},
    {"name": "8.5", "mid_suffix": "mid8p5", "eoc_suffix": "eoc8p5"},
]

season_months = "DJF"        # DJF | MAM | JJA | SON  <-- change per run

DATA_ROOT  = "/home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods"
base_path  = f"{DATA_ROOT}/combined_seasonal_datasets"
path_mask  = "/home/sleake/Masking"
output_dir = "./figures"
os.makedirs(output_dir, exist_ok=True)

print(f"Season: {season_months}\n  data: {base_path}\n  mask: {path_mask}")

Season: DJF
  data: /home/scratch/sleake/AFWA_TOTPRECIP_15min_precip_periods/combined_seasonal_datasets
  mask: /home/sleake/Masking


In [3]:
import numpy as np
from scipy.stats import mannwhitneyu

def mwu_p(hist, fut):
    h = np.asarray(hist, float); f = np.asarray(fut, float)
    h = h[np.isfinite(h)];       f = f[np.isfinite(f)]
    if h.size < 5 or f.size < 5:            return np.nan
    if np.ptp(np.concatenate([h, f])) == 0: return np.nan
    return mannwhitneyu(h, f, alternative="two-sided").pvalue

def bh_fdr(pvals, alpha=0.05):
    p = np.asarray(pvals, float)
    sig = np.zeros(p.shape, bool)
    ok = np.isfinite(p); pv = p[ok]; n = pv.size
    if n == 0: return sig
    ranked = np.sort(pv)
    passed = ranked <= alpha * np.arange(1, n + 1) / n
    if passed.any():
        sig[ok] = pv <= ranked[np.max(np.where(passed))]
    return sig

def bands_annual(dist_ds, accum_ds):
    """Per-year regional means: the concat_dim entries sharing a
    percentile_label are the individual years."""
    labels = sorted(set(dist_ds.percentile_label.values),
                    key=lambda x: float(x.split('-')[0]) if x != '0-0' else -1)
    if '0-0' in labels:
        labels.remove('0-0'); labels = ['0-0'] + labels
    dm = dist_ds.PRECIP_PERIOD.mean(dim=('south_north','west_east'), skipna=True).values
    am = accum_ds.PRECIP_SUM.mean(dim=('south_north','west_east'), skipna=True).values
    dl = np.asarray(dist_ds.percentile_label.values)
    al = np.asarray(accum_ds.percentile_label.values)
    return labels, {l: dm[dl == l] for l in labels}, {l: am[al == l] for l in labels}

def load_annual_series(region, scenario, season):
    """Seasonal store naming: <kind>_<suffix>_<season>_dataset.zarr"""
    ms, es = scenario["mid_suffix"], scenario["eoc_suffix"]
    mask = xr.open_dataset(os.path.join(path_mask, f"Masked_NCA_{region}.nc"))["mask"]
    def _open(kind, suf):
        fn = f"{kind}_{suf}_{season}_dataset.zarr"
        return xr.open_dataset(os.path.join(base_path, fn), engine="zarr").where(mask)
    out = {}
    for epoch, suf in (("hist","hist"), ("mid",ms), ("eoc",es)):
        labels, d, a = bands_annual(_open("dist", suf), _open("accum", suf))
        out[epoch] = {"labels": labels, "dist": d, "accum": a}
    return out

In [4]:
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
SCEN, METRIC, ALPHA = '8.5', 'dist', 0.05

def seasonal_panel(season, scen_name=SCEN, metric=METRIC):
    """Percent-change matrix and significance mask for one season.
    Rows are region x (Mid, EOC), matching Figure 2's row order."""
    scen = next(s for s in scenarios if s['name'] == scen_name)
    ann  = {r: load_annual_series(r, scen, season) for r in regions}
    labels = ann[regions[0]]['hist']['labels']
    keep   = [j for j, lab in enumerate(labels) if lab != '0-0']

    D = np.full((len(regions) * 2, len(labels)), np.nan)
    P = np.full_like(D, np.nan)
    for r, region in enumerate(regions):
        for e, epoch in enumerate(('mid', 'eoc')):
            for j, lab in enumerate(labels):
                h = np.asarray(ann[region]['hist'][metric][lab], float)
                f = np.asarray(ann[region][epoch][metric][lab], float)
                D[r*2+e, j] = 100.0 * (np.nanmean(f) - np.nanmean(h)) / np.nanmean(h)
                if j in keep:
                    P[r*2+e, j] = mwu_p(h, f)
    return D, bh_fdr(P, ALPHA), labels

panels = {}
for seas in SEASONS:
    D, sig, labels = seasonal_panel(seas)
    panels[seas] = (D, sig, labels)
    print(f"  {seas}: {int(sig.sum())} of {int(np.isfinite(D[:, 1:]).sum())} significant")

np.savez('seasonal_panels_rcp85_occ.npz',
         **{f"{s}_D": panels[s][0] for s in SEASONS},
         **{f"{s}_sig": panels[s][1] for s in SEASONS},
         labels=np.array(panels[SEASONS[0]][2], dtype=object), allow_pickle=True)
print("cached to seasonal_panels_rcp85_occ.npz")

  DJF: 25 of 154 significant
  MAM: 55 of 154 significant
  JJA: 131 of 154 significant
  SON: 27 of 154 significant
cached to seasonal_panels_rcp85_occ.npz


In [5]:
LABEL_MAP = {'0-0': '<0.254mm', '0-1': '0.254mm-10th', '1-2': '10th-20th',
             '2-3': '20th-30th', '3-4': '30th-40th', '4-5': '40th-50th',
             '5-6': '50th-60th', '6-7': '60th-70th', '7-8': '70th-80th',
             '8-9': '80th-90th', '9-99': '90th-99th', '99-100': '99th+'}

raw_labels = panels[SEASONS[0]][2]
percentile_labels = [LABEL_MAP.get(l, l) for l in raw_labels]
print(list(zip(raw_labels, percentile_labels)))

[('0-0', '<0.254mm'), ('0-10', '0-10'), ('10-20', '10-20'), ('20-30', '20-30'), ('30-40', '30-40'), ('40-50', '40-50'), ('50-60', '50-60'), ('60-70', '60-70'), ('70-80', '70-80'), ('80-90', '80-90'), ('90-99', '90-99'), ('99-100', '99th+')]


In [6]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

vmin, vmax = -50, 50
cmap = plt.cm.BrBG

region_abbrevs = {"Midwest": "MW", "Northeast": "NE", "NorthernGreatPlains": "NGP",
                  "Northwest": "NW", "Southeast": "SE",
                  "SouthernGreatPlains": "SGP", "Southwest": "SW"}

row_labels = []
for region in regions:
    ab = region_abbrevs.get(region, region[:2])
    row_labels += [f"{ab} Mid", f"{ab} EOC"]

fig, axes = plt.subplots(2, 2, figsize=(9.75, 10.5), sharex='col', sharey='row')
axes_flat = axes.flatten()

for idx, seas in enumerate(SEASONS):
    ax = axes_flat[idx]
    data, sig, _ = panels[seas]

    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            ax.text(j, i, f"{v:.0f}" if abs(v) >= 10 else f"{v:.1f}",
                    ha="center", va="center",
                    color='black' if sig[i, j] else '#585858',
                    fontsize=7,
                    fontweight='bold' if sig[i, j] else 'normal',
                    path_effects=[path_effects.withStroke(
                        linewidth=2 if sig[i, j] else 1.5, foreground='white')])

    ax.set_title(seas, fontsize=12)
    if idx % 2 == 0:
        ax.set_yticks(np.arange(len(row_labels)))
        ax.set_yticklabels(row_labels, fontsize=9)
    if idx >= 2:
        ax.set_xticks(np.arange(len(percentile_labels)))
        ax.set_xticklabels(percentile_labels, rotation=45, ha='right', fontsize=9)
    for y in np.arange(1.5, len(row_labels), 2):
        ax.axhline(y=y, color='black', linestyle='-', linewidth=1.0, alpha=0.7)

cax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
plt.colorbar(im, cax=cax, extend='both').set_label('Δ (%)', fontsize=12)

plt.tight_layout(rect=[0, 0, 0.92, 0.95])
for ext in ('png', 'pdf'):
    plt.savefig(os.path.join(output_dir, f'seasonal_heatmap_rcp85_occurrence.{ext}'),
                dpi=300 if ext == 'png' else None)
print("saved")
plt.show()

/tmp/ipykernel_28447/2518670755.py:49: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.92, 0.95])


saved


## Helper Functions

Core processing functions for seasonal percentile band analysis:
- `create_readable_labels()`: Convert numeric band codes to human-readable labels
- `process_percentile_bands()`: Calculate spatial averages with equal grid cell weighting
- `safe_percent_change()`: Compute percent change with division-by-zero protection
- `load_and_process_data()`: Full pipeline for loading, masking, and processing seasonal regional data

In [43]:
# Helper Functions

def create_readable_labels(labels):
    """
    Create human-readable labels for the percentile bands.
    
    Args:
        labels: List of percentile band codes (e.g., '0-0', '9-99')
    
    Returns:
        List of readable labels (e.g., 'None-Trace', '90th-99th')
    """
    readable_labels = []
    for label in labels:
        if label == '0-0':
            readable_labels.append('None-Trace')
        else:
            parts = label.split('-')
            if len(parts) == 2:
                # Map numeric codes to percentile labels
                label_map = {
                    ('0', '1'): 'Trace-10th',
                    ('1', '2'): '10th-20th',
                    ('2', '3'): '20th-30th',
                    ('3', '4'): '30th-40th',
                    ('4', '5'): '40th-50th',
                    ('5', '6'): '50th-60th',
                    ('6', '7'): '60th-70th',
                    ('7', '8'): '70th-80th',
                    ('8', '9'): '80th-90th',
                    ('9', '99'): '90th-99th',
                    ('99', '0'): '99th+' #yes, this format is weird... it is just a label
                }
                readable_labels.append(label_map.get(tuple(parts), f'{parts[0]}th-{parts[1]}th'))
    return readable_labels


def process_percentile_bands(dist_ds, accum_ds):
    """
    Process percentile bands using statistically sound spatial averaging:
    1. Calculate temporal sum for each grid cell
    2. Compute spatial average across all grid cells (equal weighting)
    
    Args:
        dist_ds: Distribution dataset (PRECIP_PERIOD counts)
        accum_ds: Accumulation dataset (PRECIP_SUM totals)
    
    Returns:
        Tuple of (percentile_labels, distribution_means, accumulation_means)
    """
    # Get unique percentile labels and sort (with '0-0' first)
    percentile_labels = sorted(
        list(set(dist_ds.percentile_label.values)), 
        key=lambda x: float(x.split('-')[0]) if x != '0-0' else -1
    )
    
    if '0-0' in percentile_labels:
        percentile_labels.remove('0-0')
        percentile_labels = ['0-0'] + percentile_labels
    
    # Calculate means for each percentile band
    dist_means = {}
    accum_means = {}
    
    for label in percentile_labels:
        # Select data for current percentile band
        dist_band = dist_ds.isel(
            concat_dim=[i for i, plabel in enumerate(dist_ds.percentile_label.values) 
                       if plabel == label]
        )
        accum_band = accum_ds.isel(
            concat_dim=[i for i, plabel in enumerate(accum_ds.percentile_label.values) 
                        if plabel == label]
        )
        
        # Temporal sum over concat_dim, then spatial mean
        dist_sum = dist_band.PRECIP_PERIOD.sum(dim='concat_dim')
        accum_sum = accum_band.PRECIP_SUM.sum(dim='concat_dim')
        
        # Spatial mean (excluding NaN from mask)
        dist_means[label] = float(dist_sum.mean(skipna=True).values)
        accum_means[label] = float(accum_sum.mean(skipna=True).values)
    
    return percentile_labels, dist_means, accum_means


def safe_percent_change(new_val, old_val, max_val=1000):
    """
    Calculate percent change with safety checks.
    
    Args:
        new_val: Future value
        old_val: Historical baseline value
        max_val: Maximum absolute percent change to return (clipping threshold)
    
    Returns:
        Percent change, or 0 if baseline is zero/NaN
    """
    if old_val == 0 or np.isnan(old_val) or np.isnan(new_val):
        return 0
    
    percent_change = ((new_val / old_val) - 1) * 100
    
    if np.isinf(percent_change) or np.isnan(percent_change):
        return 0
    
    return np.clip(percent_change, -max_val, max_val)


def load_and_process_data(region, scenario):
    """
    Load and process seasonal data for a specific region and scenario.
    
    Pipeline:
    1. Load historical, mid-century, and end-of-century seasonal datasets
    2. Apply regional mask
    3. Process percentile bands (temporal sum → spatial average)
    4. Calculate percent changes vs. historical baseline
    
    Args:
        region: Region name (e.g., 'Midwest')
        scenario: Scenario dict with name and file suffixes
    
    Returns:
        Dict with readable labels and DataFrames of percent changes
    """
    scenario_name = scenario["name"]
    mid_suffix = scenario["mid_suffix"]
    eoc_suffix = scenario["eoc_suffix"]
    
    # Define paths for seasonal distribution and accumulation datasets
    hist_dist_path = os.path.join(base_path, f"dist_hist_{season_months}_dataset.zarr")
    mid_dist_path = os.path.join(base_path, f"dist_{mid_suffix}_{season_months}_dataset.zarr")
    eoc_dist_path = os.path.join(base_path, f"dist_{eoc_suffix}_{season_months}_dataset.zarr")

    hist_accum_path = os.path.join(base_path, f"accum_hist_{season_months}_dataset.zarr")
    mid_accum_path = os.path.join(base_path, f"accum_{mid_suffix}_{season_months}_dataset.zarr")
    eoc_accum_path = os.path.join(base_path, f"accum_{eoc_suffix}_{season_months}_dataset.zarr")
    
    print(f"  Loading mask for {region}...")
    # Load regional mask
    mask = xr.open_dataset(os.path.join(path_mask, f"Masked_NCA_{region}.nc"))
    
    print(f"  Loading {season_months} datasets for RCP{scenario_name}...")
    # Load all epoch datasets
    hist_dist = xr.open_dataset(hist_dist_path, engine="zarr")
    mid_dist = xr.open_dataset(mid_dist_path, engine="zarr")
    eoc_dist = xr.open_dataset(eoc_dist_path, engine="zarr")

    hist_accum = xr.open_dataset(hist_accum_path, engine="zarr")
    mid_accum = xr.open_dataset(mid_accum_path, engine="zarr")
    eoc_accum = xr.open_dataset(eoc_accum_path, engine="zarr")
    
    print(f"  Applying {region} mask...")
    # Apply mask to all datasets
    hist_dist = hist_dist.where(mask["mask"])
    mid_dist = mid_dist.where(mask["mask"])
    eoc_dist = eoc_dist.where(mask["mask"])

    hist_accum = hist_accum.where(mask["mask"])
    mid_accum = mid_accum.where(mask["mask"])
    eoc_accum = eoc_accum.where(mask["mask"])
    
    print(f"  Processing percentile bands...")
    # Process each epoch
    hist_labels, hist_dist_totals, hist_accum_totals = process_percentile_bands(hist_dist, hist_accum)
    mid_labels, mid_dist_totals, mid_accum_totals = process_percentile_bands(mid_dist, mid_accum)
    eoc_labels, eoc_dist_totals, eoc_accum_totals = process_percentile_bands(eoc_dist, eoc_accum)
    
    # Ensure common percentile bands across all epochs
    common_labels = sorted(
        list(set(hist_labels) & set(mid_labels) & set(eoc_labels)), 
        key=lambda x: float(x.split('-')[0]) if x != '0-0' else -1
    )

    if '0-0' in common_labels:
        common_labels.remove('0-0')
        common_labels = ['0-0'] + common_labels
    
    # Create readable labels
    readable_labels = create_readable_labels(common_labels)
    
    # Calculate percent changes with safety checks
    percent_change_mid_dist = pd.DataFrame({
        'Percentile Band': common_labels,
        'Readable Label': readable_labels,
        'Percent Change': [safe_percent_change(mid_dist_totals[label], hist_dist_totals[label]) 
                          for label in common_labels]
    })
    
    percent_change_eoc_dist = pd.DataFrame({
        'Percentile Band': common_labels,
        'Readable Label': readable_labels,
        'Percent Change': [safe_percent_change(eoc_dist_totals[label], hist_dist_totals[label]) 
                          for label in common_labels]
    })
    
    percent_change_mid_accum = pd.DataFrame({
        'Percentile Band': common_labels,
        'Readable Label': readable_labels,
        'Percent Change': [safe_percent_change(mid_accum_totals[label], hist_accum_totals[label]) 
                          for label in common_labels]
    })
    
    percent_change_eoc_accum = pd.DataFrame({
        'Percentile Band': common_labels,
        'Readable Label': readable_labels,
        'Percent Change': [safe_percent_change(eoc_accum_totals[label], hist_accum_totals[label]) 
                          for label in common_labels]
    })
    
    return {
        'readable_labels': readable_labels,
        'mid_dist': percent_change_mid_dist,
        'eoc_dist': percent_change_eoc_dist,
        'mid_accum': percent_change_mid_accum,
        'eoc_accum': percent_change_eoc_accum
    }

In [44]:
# Data Collection

print(f"Collecting {season_months} data using improved spatial averaging methodology...")
all_data = {}

for region in regions:
    print(f"\nProcessing region: {region}")
    all_data[region] = {}
    
    for scenario in scenarios:
        print(f"  Processing scenario: RCP{scenario['name']}")
        all_data[region][scenario['name']] = load_and_process_data(region, scenario)

print(f"\n{season_months} data collection complete!")


Processing region: Midwest
  Processing scenario: RCP4.5
  Loading mask for Midwest...
  Loading DJF datasets for RCP4.5...
  Applying Midwest mask...
  Processing percentile bands...
  Processing scenario: RCP8.5
  Loading mask for Midwest...
  Loading DJF datasets for RCP8.5...
  Applying Midwest mask...
  Processing percentile bands...

Processing region: Northeast
  Processing scenario: RCP4.5
  Loading mask for Northeast...
  Loading DJF datasets for RCP4.5...
  Applying Northeast mask...
  Processing percentile bands...
  Processing scenario: RCP8.5
  Loading mask for Northeast...
  Loading DJF datasets for RCP8.5...
  Applying Northeast mask...
  Processing percentile bands...

Processing region: NorthernGreatPlains
  Processing scenario: RCP4.5
  Loading mask for NorthernGreatPlains...
  Loading DJF datasets for RCP4.5...
  Applying NorthernGreatPlains mask...
  Processing percentile bands...
  Processing scenario: RCP8.5
  Loading mask for NorthernGreatPlains...
  Loading DJF

## Small Multiples Visualization

Generate individual bar chart figures for each region showing seasonal patterns:
- 8 subplots per region (2 scenarios × 4 metrics)
- Fixed y-axis limits (-40% to 90%) for cross-region comparison
- Grid lines at 10% increments with emphasized zero line
- Color coding: dark gray (increases), light gray (decreases)
- Filenames include season suffix for organization

In [45]:
# Create Small Multiples with One Row Per Figure

# Fixed axis limits for all plots (enables cross-region comparison)
YMIN = -40
YMAX = 90

print(f"\nCreating {season_months} small multiples with FIXED y-axis limits: [{YMIN}, {YMAX}]")

# Create one figure per region
for r, region in enumerate(regions):
    fig = plt.figure(figsize=(16, 2))
    row_axes = []
    
    # Create 8 subplots (2 scenarios × 4 plot types)
    for j, scenario in enumerate(scenarios):
        base_col = j * 4
        data = all_data[region][scenario['name']]
        
        # Plot configuration
        plot_types = ['mid_dist', 'eoc_dist', 'mid_accum', 'eoc_accum']
        titles = [
            f"Mid RCP{scenario['name']} Occur.",
            f"EOC RCP{scenario['name']} Occur.",
            f"Mid RCP{scenario['name']} Accum.",
            f"EOC RCP{scenario['name']} Accum."
        ]
        
        for i, (plot_type, title) in enumerate(zip(plot_types, titles)):
            pos = base_col + i + 1
            ax = fig.add_subplot(1, 8, pos)
            row_axes.append(ax)
            
            # Set fixed limits BEFORE plotting
            ax.set_ylim(YMIN, YMAX)
            
            # Get data and plot bars
            plot_data = data[plot_type]
            changes = plot_data['Percent Change'].values
            colors = ['darkgray' if val >= 0 else 'lightgray' for val in changes]
            bars = ax.bar(range(len(changes)), changes, color=colors, 
                         edgecolor='black', linewidth=0.5)
            
            # Add grid lines
            for y in range(YMIN, YMAX+1, 10):
                if y != 0:
                    ax.axhline(y=y, color='lightgray', linestyle='-', 
                              linewidth=0.5, alpha=0.5, zorder=0)
            
            # Emphasize zero line
            ax.axhline(y=0, color='black', linestyle='-', 
                      linewidth=0.8, alpha=0.7, zorder=1)
            
            # Title for first region only
            if r == 0:
                ax.set_title(title, fontsize=8, pad=4)
            
            # Remove spines
            for spine in ax.spines.values():
                spine.set_visible(False)
            
            # X-axis labels (show 5 evenly spaced for readability)
            n_labels = len(plot_data['Readable Label'])
            if n_labels > 10:
                indices = np.linspace(0, n_labels-1, 5).astype(int)
                labels = [plot_data['Readable Label'].iloc[idx] for idx in indices]
                ax.set_xticks(indices)
                ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
            else:
                ax.set_xticks(range(n_labels))
                ax.set_xticklabels(plot_data['Readable Label'], rotation=45, ha='right', fontsize=6)
            
            # Y-axis labels only for first subplot
            if i == 0:
                ax.set_yticks(range(YMIN, YMAX+1, 10))
                ax.tick_params(axis='y', labelsize=6)
            else:
                ax.set_yticks([])
            
            # Region label on left side
            if i == 0:
                ax.text(-0.5, 0.5, region, transform=ax.transAxes, 
                       va='center', ha='right', fontsize=10, fontweight='bold',
                       rotation=90)
    
    # Verify consistent axis limits
    for ax in row_axes:
        assert ax.get_ylim() == (YMIN, YMAX), f"Axis y-limits are wrong: {ax.get_ylim()}"
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.25)
    
    # Save individual region figure with season suffix
    output_path = os.path.join(output_dir, f"{region}_precipitation_changes_{season_months}.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

print(f"{season_months} small multiples created successfully!")


Creating DJF small multiples with FIXED y-axis limits: [-40, 90]
DJF small multiples created successfully!


In [46]:
# Combine All Region Figures into One Master Figure

print(f"\nCombining all {season_months} region figures into master figure...")

# Create master figure
master_fig = plt.figure(figsize=(12, 24))

# Add titles and labels
master_fig.suptitle(f'{season_months} Percent Change in Precipitation by Percentile Interval Across Regions and Scenarios', 
                   fontsize=14, y=0.98)
master_fig.text(0.02, 0.5, 'Percent Change (%)', va='center', rotation=90, fontsize=12)
master_fig.text(0.5, 0.02, 'Percentile Interval (from None-Trace to 99th+)', ha='center', fontsize=12)

# Load and place each region's figure
for r, region in enumerate(regions):
    region_path = os.path.join(output_dir, f"{region}_precipitation_changes_{season_months}.png")
    region_fig = plt.imread(region_path)
    
    ax = master_fig.add_subplot(len(regions), 1, r+1)
    ax.imshow(region_fig)
    ax.axis('off')

# Adjust layout and save
plt.tight_layout(rect=[0.03, 0.03, 0.97, 0.95])

output_path = os.path.join(output_dir, f"precipitation_percent_change_small_multiples_{season_months}.png")
plt.savefig(output_path, dpi=300)

output_path = os.path.join(output_dir, f"precipitation_percent_change_small_multiples_{season_months}.pdf")
plt.savefig(output_path)

print(f"{season_months} master figure saved successfully!")
print(f"All plots use identical y-axis limits: [{YMIN}, {YMAX}]")


Combining all DJF region figures into master figure...
DJF master figure saved successfully!
All plots use identical y-axis limits: [-40, 90]


## Seasonal Heatmap Visualization

Generate 2×2 heatmap grid for seasonal patterns:
- **Layout**: RCP4.5/8.5 (rows) × Occurrences/Accumulation (columns)
- **Color scale**: Diverging BrBG colormap (-50% to +50%)
- **Annotations**: Percent change values with white outline for readability
- **Region labels**: Abbreviated (e.g., MW, NE) with Mid/EOC distinction
- **Output**: PNG (300 dpi) and PDF formats with season suffix

In [47]:
# Heatmap Visualization with Diverging Colormap - 2×2 Layout

print(f"Creating {season_months} heatmap visualization with improved text annotations...")

# Get percentile labels from any region/scenario
percentile_labels = all_data[regions[0]][scenarios[0]['name']]['mid_dist']['Readable Label'].values

# Set fixed color scale limits
vmin = -50  
vmax = 50

# Create 2×2 figure
fig, axes = plt.subplots(2, 2, figsize=(9.75, 10.5), sharex='col', sharey='row')
axes_flat = axes.flatten()

# Use BrBG colormap for good grayscale rendering
cmap = plt.cm.BrBG

# Define plot configuration
plot_configs = [
    {'scenario': '4.5', 'data_type': 'dist', 'title': 'RCP4.5 Occurrences', 'ax_idx': 0},
    {'scenario': '4.5', 'data_type': 'accum', 'title': 'RCP4.5 Accumulation', 'ax_idx': 1},
    {'scenario': '8.5', 'data_type': 'dist', 'title': 'RCP8.5 Occurrences', 'ax_idx': 2},
    {'scenario': '8.5', 'data_type': 'accum', 'title': 'RCP8.5 Accumulation', 'ax_idx': 3}
]

# Region abbreviations
region_abbrevs = {
    "Midwest": "MW",
    "Northeast": "NE", 
    "NorthernGreatPlains": "NGP",
    "Northwest": "NW",
    "Southeast": "SE",
    "SouthernGreatPlains": "SGP",
    "Southwest": "SW"
}

# Create each heatmap
for config in plot_configs:
    ax = axes_flat[config['ax_idx']]
    
    # Prepare data array (rows: regions × 2 epochs, columns: percentile bands)
    data = np.zeros((len(regions) * 2, len(percentile_labels)))
    row_labels = []
    
    for r_idx, region in enumerate(regions):
        mid_row = r_idx * 2
        eoc_row = r_idx * 2 + 1
        
        abbrev = region_abbrevs.get(region, region[:2])
        row_labels.append(f"{abbrev} Mid")
        row_labels.append(f"{abbrev} EOC")
        
        # Get data keys
        mid_key = f"mid_{config['data_type']}"
        eoc_key = f"eoc_{config['data_type']}"
        
        if config['scenario'] in all_data[region]:
            data[mid_row, :] = all_data[region][config['scenario']][mid_key]['Percent Change'].values
            data[eoc_row, :] = all_data[region][config['scenario']][eoc_key]['Percent Change'].values
    
    # Create heatmap
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')
    
    # Add text annotations with white outline for readability
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data[i, j]
            text = f"{value:.0f}" if abs(value) >= 10 else f"{value:.1f}"
            
            ax.text(j, i, text, ha="center", va="center", 
                    color='black', fontsize=7, fontweight='bold',
                    path_effects=[
                        path_effects.withStroke(linewidth=2, foreground='white')
                    ])
    
    # Set title
    ax.set_title(config['title'], fontsize=12)
    
    # Y-axis labels (left column only)
    if config['ax_idx'] % 2 == 0:
        ax.set_yticks(np.arange(len(row_labels)))
        ax.set_yticklabels(row_labels, fontsize=9)
    
    # X-axis labels (bottom row only)
    if config['ax_idx'] >= 2:
        ax.set_xticks(np.arange(len(percentile_labels)))
        ax.set_xticklabels(percentile_labels, rotation=45, ha='right', fontsize=9)
    
    # Add lines to separate region pairs
    for y in np.arange(1.5, len(row_labels), 2):
        ax.axhline(y=y, color='black', linestyle='-', linewidth=1.0, alpha=0.7)

# Add colorbar
cax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
cbar = plt.colorbar(im, cax=cax, extend='both')
cbar.set_label('Δ (%)', fontsize=12)

# Save figure with season suffix
plt.tight_layout(rect=[0, 0, 0.92, 0.95])

output_path = os.path.join(output_dir, f'precipitation_heatmap_annotated_{season_months}.png')
plt.savefig(output_path, dpi=300)

output_path = os.path.join(output_dir, f'precipitation_heatmap_annotated_{season_months}.pdf')
plt.savefig(output_path)

print(f"{season_months} annotated heatmap visualization complete")

Creating DJF heatmap visualization with improved text annotations...


/tmp/ipykernel_15688/834267363.py:100: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.92, 0.95])


DJF annotated heatmap visualization complete


In [48]:
# Significance testing -- seasonal
# Sample: the concat_dim entries sharing a percentile_label are the individual
# years, so a spatial mean per year gives one value per year per band.

def mwu_p(hist, fut):
    h = np.asarray(hist, float); f = np.asarray(fut, float)
    h = h[np.isfinite(h)];       f = f[np.isfinite(f)]
    if h.size < 5 or f.size < 5:            return np.nan
    if np.ptp(np.concatenate([h, f])) == 0: return np.nan
    return mannwhitneyu(h, f, alternative="two-sided").pvalue


def bh_fdr(pvals, alpha=0.05):
    """Benjamini-Hochberg. NaNs are excluded from the family."""
    p = np.asarray(pvals, float)
    sig = np.zeros(p.shape, bool)
    ok = np.isfinite(p); pv = p[ok]; n = pv.size
    if n == 0: return sig
    ranked = np.sort(pv)
    passed = ranked <= alpha * np.arange(1, n + 1) / n
    if passed.any():
        sig[ok] = pv <= ranked[np.max(np.where(passed))]
    return sig


# def bands_annual(dist_ds, accum_ds):
#     """Per-year regional means, i.e. process_percentile_bands without the
#     sum over concat_dim."""
#     labels = sorted(set(dist_ds.percentile_label.values),
#                     key=lambda x: float(x.split('-')[0]) if x != '0-0' else -1)
#     if '0-0' in labels:
#         labels.remove('0-0'); labels = ['0-0'] + labels

#     d, a = {}, {}
#     for lab in labels:
#         di = [i for i, p in enumerate(dist_ds.percentile_label.values)  if p == lab]
#         ai = [i for i, p in enumerate(accum_ds.percentile_label.values) if p == lab]
#         d[lab] = dist_ds.isel(concat_dim=di).PRECIP_PERIOD.mean(
#             dim=('south_north', 'west_east'), skipna=True).values
#         a[lab] = accum_ds.isel(concat_dim=ai).PRECIP_SUM.mean(
#             dim=('south_north', 'west_east'), skipna=True).values
#     return labels, d, a

def bands_annual(dist_ds, accum_ds):
    labels = sorted(set(dist_ds.percentile_label.values),
                    key=lambda x: float(x.split('-')[0]) if x != '0-0' else -1)
    if '0-0' in labels:
        labels.remove('0-0'); labels = ['0-0'] + labels

    # one spatial reduction over the whole concat_dim, then split by label
    dm = dist_ds.PRECIP_PERIOD.mean(dim=('south_north','west_east'), skipna=True).values
    am = accum_ds.PRECIP_SUM.mean(dim=('south_north','west_east'), skipna=True).values
    dl = np.asarray(dist_ds.percentile_label.values)
    al = np.asarray(accum_ds.percentile_label.values)

    d = {lab: dm[dl == lab] for lab in labels}
    a = {lab: am[al == lab] for lab in labels}
    return labels, d, a


def load_annual_series(region, scenario):
    """Seasonal paths: dist_<suffix>_<season>_dataset.zarr"""
    ms, es = scenario["mid_suffix"], scenario["eoc_suffix"]
    mask = xr.open_dataset(os.path.join(path_mask, f"Masked_NCA_{region}.nc"))

    def _open(kind, suf):
        fn = f"{kind}_{suf}_{season_months}_dataset.zarr"
        return xr.open_dataset(os.path.join(base_path, fn),
                               engine="zarr").where(mask["mask"])

    out = {}
    for epoch, suf in (("hist", "hist"), ("mid", ms), ("eoc", es)):
        labels, d, a = bands_annual(_open("dist", suf), _open("accum", suf))
        out[epoch] = {"labels": labels, "dist": d, "accum": a}
    return out

def load_all_regions(scenario):
    """Open each store once; apply all seven masks to it before moving on."""
    ms, es = scenario["mid_suffix"], scenario["eoc_suffix"]
    masks = {r: xr.open_dataset(os.path.join(path_mask, f"Masked_NCA_{r}.nc"))["mask"]
             for r in regions}

    out = {r: {} for r in regions}
    for epoch, suf in (("hist", "hist"), ("mid", ms), ("eoc", es)):
        d_ds = xr.open_dataset(os.path.join(
            base_path, f"dist_{suf}_{season_months}_dataset.zarr"), engine="zarr")
        a_ds = xr.open_dataset(os.path.join(
            base_path, f"accum_{suf}_{season_months}_dataset.zarr"), engine="zarr")
        for r in regions:
            labels, d, a = bands_annual(d_ds.where(masks[r]), a_ds.where(masks[r]))
            out[r][epoch] = {"labels": labels, "dist": d, "accum": a}
        d_ds.close(); a_ds.close()
        print(f"  {epoch} done")
    return out

annual = {}
for s in scenarios:
    print(f"RCP{s['name']}")
    per_region = load_all_regions(s)
    for r in regions:
        annual.setdefault(r, {})[s["name"]] = per_region[r]

# annual = {r: {s["name"]: load_annual_series(r, s) for s in scenarios}
#           for r in regions}

# from itertools import product
# from tqdm import tqdm

# annual = {r: {} for r in regions}
# for r, s in tqdm(list(product(regions, scenarios)), desc="loading"):
#     annual[r][s["name"]] = load_annual_series(r, s)

# sanity: how many years does this season actually give us?
print("Starting sig test")
_l = annual[regions[0]][scenarios[0]['name']]['hist']['labels'][-1]
_n = np.size(annual[regions[0]][scenarios[0]['name']]['hist']['dist'][_l])
print(f"{season_months}: {_n} values per sample")
if _n < 5:
    raise ValueError("no usable year dimension along concat_dim")

ALPHA = 0.05
DROP_BANDS = {'0-0'}          # sub-threshold: shown, not tested

pvals, sigmask = {}, {}
for scen in scenarios:
    sn = scen["name"]
    for metric in ("dist", "accum"):
        labels = annual[regions[0]][sn]["hist"]["labels"]
        keep   = [j for j, lab in enumerate(labels) if lab not in DROP_BANDS]

        P = np.full((len(regions) * 2, len(labels)), np.nan)   # region-major, Mid then EOC
        for r, region in enumerate(regions):
            for e, epoch in enumerate(("mid", "eoc")):
                for j in keep:
                    lab = labels[j]
                    P[r * 2 + e, j] = mwu_p(annual[region][sn]["hist"][metric][lab],
                                            annual[region][sn][epoch][metric][lab])

        pvals[(sn, metric)]   = P
        sigmask[(sn, metric)] = bh_fdr(P, ALPHA)
        print(f"  RCP{sn} {metric}: {sigmask[(sn,metric)].sum()}/"
              f"{np.isfinite(P).sum()} significant")

RCP4.5
  hist done
  mid done
  eoc done
RCP8.5
  hist done
  mid done
  eoc done
Starting sig test
DJF: 15 values per sample
  RCP4.5 dist: 33/154 significant
  RCP4.5 accum: 31/154 significant
  RCP8.5 dist: 25/154 significant
  RCP8.5 accum: 22/154 significant


In [49]:
print(f"Creating {season_months} heatmap with significance...")

percentile_labels = all_data[regions[0]][scenarios[0]['name']]['mid_dist']['Readable Label'].values
LABEL_FIX = {'None-Trace': '<0.254mm', '0th-10th': "0.254mm-10th", 'Trace-10th': '0.254mm-10th'}

vmin, vmax = -50, 50
fig, axes = plt.subplots(2, 2, figsize=(9.75, 10.5), sharex='col', sharey='row')
axes_flat = axes.flatten()
cmap = plt.cm.BrBG

plot_configs = [
    {'scenario': '4.5', 'data_type': 'dist',  'title': 'RCP4.5 Occurrences',  'ax_idx': 0},
    {'scenario': '4.5', 'data_type': 'accum', 'title': 'RCP4.5 Accumulation', 'ax_idx': 1},
    {'scenario': '8.5', 'data_type': 'dist',  'title': 'RCP8.5 Occurrences',  'ax_idx': 2},
    {'scenario': '8.5', 'data_type': 'accum', 'title': 'RCP8.5 Accumulation', 'ax_idx': 3},
]

region_abbrevs = {"Midwest": "MW", "Northeast": "NE", "NorthernGreatPlains": "NGP",
                  "Northwest": "NW", "Southeast": "SE",
                  "SouthernGreatPlains": "SGP", "Southwest": "SW"}

for config in plot_configs:
    ax = axes_flat[config['ax_idx']]
    data = np.zeros((len(regions) * 2, len(percentile_labels)))
    row_labels = []

    for r_idx, region in enumerate(regions):
        mid_row, eoc_row = r_idx * 2, r_idx * 2 + 1
        abbrev = region_abbrevs.get(region, region[:2])
        row_labels += [f"{abbrev} Mid", f"{abbrev} EOC"]
        if config['scenario'] in all_data[region]:
            d = all_data[region][config['scenario']]
            data[mid_row, :] = d[f"mid_{config['data_type']}"]['Percent Change'].values
            data[eoc_row, :] = d[f"eoc_{config['data_type']}"]['Percent Change'].values

    key = (config['scenario'], config['data_type'])
    if key not in sigmask:
        raise KeyError(f"{key} not in sigmask; have {sorted(sigmask)} "
                       "-- rerun the significance cell")
    if sigmask[key].shape != data.shape:
        raise ValueError(f"sigmask{key} {sigmask[key].shape} vs data {data.shape}")
    sig = sigmask[key]

    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, aspect='equal')

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            ax.text(j, i, f"{v:.0f}" if abs(v) >= 10 else f"{v:.1f}",
                    ha="center", va="center",
                    color='black' if sig[i, j] else '#585858',
                    fontsize=7,
                    fontweight='bold' if sig[i, j] else 'normal',
                    path_effects=[path_effects.withStroke(
                        linewidth=2 if sig[i, j] else 1.5, foreground='white')])

    ax.set_title(config['title'], fontsize=12)
    if config['ax_idx'] % 2 == 0:
        ax.set_yticks(np.arange(len(row_labels)))
        ax.set_yticklabels(row_labels, fontsize=9)
    if config['ax_idx'] >= 2:
        ax.set_xticks(np.arange(len(percentile_labels)))
        ax.set_xticklabels([LABEL_FIX.get(l, l) for l in percentile_labels],
                           rotation=45, ha='right', fontsize=9)
    for y in np.arange(1.5, len(row_labels), 2):
        ax.axhline(y=y, color='black', linestyle='-', linewidth=1.0, alpha=0.7)

    print(f"  {config['title']}: {int(sig.sum())} of {sig.size} significant")

cax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
plt.colorbar(im, cax=cax, extend='both').set_label('Δ (%)', fontsize=12)
#fig.suptitle(f'{season_months}', fontsize=13, y=0.98)

plt.tight_layout(rect=[0, 0, 0.92, 0.95])
for ext in ('png', 'pdf'):
    plt.savefig(os.path.join(output_dir,
                f'precipitation_heatmap_annotated_{season_months}.{ext}'),
                dpi=300 if ext == 'png' else None)
print("done")

Creating DJF heatmap with significance...
  RCP4.5 Occurrences: 33 of 168 significant
  RCP4.5 Accumulation: 31 of 168 significant
  RCP8.5 Occurrences: 25 of 168 significant
  RCP8.5 Accumulation: 22 of 168 significant


/tmp/ipykernel_15688/3669090303.py:74: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 0.92, 0.95])


done


In [ ]:
# All seasons for one accum/occur and rcp

In [ ]:
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
SCEN, METRIC = '8.5', 'dist'

def seasonal_panel(seas, scen_name=SCEN, metric=METRIC):
    """Percent-change matrix and significance mask for one season.
    Rows are region x (Mid, EOC), matching Figure 2."""
    global season_months
    season_months = seas                      # load_annual_series reads this
    scen = next(s for s in scenarios if s['name'] == scen_name)

    ann    = {r: load_annual_series(r, scen) for r in regions}
    labels = ann[regions[0]]['hist']['labels']
    keep   = [j for j, lab in enumerate(labels) if lab != '0-0']

    D = np.full((len(regions) * 2, len(labels)), np.nan)
    P = np.full_like(D, np.nan)
    for r, region in enumerate(regions):
        for e, epoch in enumerate(('mid', 'eoc')):
            for j, lab in enumerate(labels):
                h = np.asarray(ann[region]['hist'][metric][lab], float)
                f = np.asarray(ann[region][epoch][metric][lab], float)
                D[r*2+e, j] = 100.0 * (np.nanmean(f) - np.nanmean(h)) / np.nanmean(h)
                if j in keep:
                    P[r*2+e, j] = mwu_p(h, f)
    return D, bh_fdr(P, 0.05), labels

panels = {}
for seas in SEASONS:
    panels[seas] = seasonal_panel(seas)
    print(f"  {seas}: {int(panels[seas][1].sum())} significant")

In [ ]:
plot_configs = [{'season': s, 'title': s, 'ax_idx': i}
                for i, s in enumerate(SEASONS)]

# inside the loop, replacing the data/sig assembly:
    data, sig, labels = panels[config['season']]
    percentile_labels = [LABEL_FIX.get(l, l) for l in
                         make_readable(labels)]   # your existing label maker